## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn", "pandas", "numpy", "matplotlib", "scipy", "tabulate"])

## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import stats
from tabulate import tabulate
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, matthews_corrcoef

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# ── Seed Range ────────────────────────────────────────────────────────────────
SEEDS         = list(range(2026, 2056))   # 30 seeds: 2026–2055 inclusive
N_RUNS        = len(SEEDS)

# ── C grid for LogisticRegression CV ─────────────────────────────────────────
# LogisticRegression uses C = 1/alpha (inverse regularisation strength).
# Higher C = less regularisation. Previous RidgeClassifier consistently
# selected alpha=100 (very high regularisation), which over-suppressed the
# LLM coefficient. This refined grid adds resolution at the high end and
# covers a wider range to let CV find the true optimum.
# NOTE: We tune C directly (not alpha) to stay native to LogisticRegression.
CV_FOLDS      = 5
C_GRID        = [0.00001, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5,
                 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]

ECE_BINS      = 10
P_ALPHA       = 0.05   # significance threshold

# ── Column names ──────────────────────────────────────────────────────────────
ENCODER_COLS  = ["indobert-p1", "indobertweet", "mdeberta-v3"]
LLM_COL       = "gemini-3-flash-preview"
FEATURE_NAMES = ENCODER_COLS + [LLM_COL]
LABEL_COL     = "label"

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path(r"*insert path here*")
OUT_DIR       = Path(r"*insert path here*")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Per-seed file templates (format with seed integer)
OOF_TRAIN_TEMPLATE  = str(PROJECT_ROOT / "oof_results" / "oof_train_{seed}.csv")
PROBS_TEST_TEMPLATE = str(PROJECT_ROOT / "oof_results" / "probs_test_{seed}.csv")

# Static files loaded ONCE outside the loop
LLM_TRAIN_NPY   = Path(r"*insert path here*")
LLM_RAND_NPY    = Path(r"*insert path here*/llm_rand_preds.npy")
LLM_TEST_NPY    = Path(r"*insert path here*/llm_test_preds.npy")
TEST_LABELS_CSV  = Path(r"*insert path here*/datd_test.csv")
TRAIN_LABELS_CSV = Path(r"*insert path here*/datd_train.csv")
RAND_LABELS_CSV  = Path(r"*insert path here*/datd_rand_cleaned.csv")

logger.info("Seeds  : %d–%d (%d runs)", SEEDS[0], SEEDS[-1], N_RUNS)
logger.info("OutDir : %s", OUT_DIR)

## 2. Shared Utility Functions

In [ ]:
def expected_calibration_error(
    y_true: np.ndarray,
    y_conf: np.ndarray,
    y_pred: np.ndarray,
    n_bins: int = ECE_BINS,
) -> float:
    r"""ECE = Σ (|Bm|/n) * |acc(Bm) − conf(Bm)|  over M equal-width bins."""
    y_true = np.asarray(y_true).ravel()
    y_conf = np.asarray(y_conf).ravel()
    y_pred = np.asarray(y_pred).ravel()
    assert y_true.shape == y_conf.shape == y_pred.shape, \
        f"ECE shape mismatch: y_true={y_true.shape}, y_conf={y_conf.shape}, y_pred={y_pred.shape}"
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    n_total   = len(y_true)
    ece       = 0.0
    for low, high in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_conf >= low) & (y_conf <= high) if high == 1.0 \
               else (y_conf >= low) & (y_conf < high)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n_total) * abs(
            (y_pred[mask] == y_true[mask]).mean() - y_conf[mask].mean()
        )
    return float(ece)


def compute_metrics(y_true, y_pred, y_conf) -> dict:
    """Return macro-F1, MCC, and ECE. All inputs are ravelled to 1D defensively."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    y_conf = np.asarray(y_conf).ravel()
    assert y_true.shape == y_pred.shape == y_conf.shape, \
        f"compute_metrics shape mismatch: y_true={y_true.shape}, "\
        f"y_pred={y_pred.shape}, y_conf={y_conf.shape}"
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc":      float(matthews_corrcoef(y_true, y_pred)),
        "ece":      float(expected_calibration_error(y_true, y_conf, y_pred)),
    }


def tune_C(
    X: np.ndarray,
    y: np.ndarray,
    c_values: list = C_GRID,
    cv: int = CV_FOLDS,
    seed: int = 42,
) -> float:
    """
    Grid-search the best LogisticRegression C via stratified k-fold CV (macro-F1).
    C = 1/regularisation_strength. Larger C = less regularisation.
    Returns the best C value.
    """
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)
    y   = np.asarray(y).ravel().astype(np.int32)
    best_C, best_score = c_values[0], -np.inf
    for C in c_values:
        clf   = LogisticRegression(C=C, max_iter=1000, random_state=seed, solver="lbfgs")
        score = cross_val_score(clf, X, y, cv=skf, scoring="f1_macro").mean()
        if score > best_score:
            best_score, best_C = score, C
    return best_C


def fit_predict_logistic(X_train, y_train, X_test, C, seed):
    """
    Fit a LogisticRegression meta-learner, return (hard_preds, probs, coef).

    Advantages over RidgeClassifier:
      - Directly optimises log-loss (classification objective, not MSE)
      - predict_proba() gives natively calibrated probabilities — no sigmoid hack needed
      - C parameter is interpretable: C=1/alpha, so smaller C = more regularisation

    StandardScaler is fit on X_train and applied to X_test — no leakage.
    """
    y_train_bin = np.asarray(y_train).ravel().astype(np.int32)
    assert set(np.unique(y_train_bin)).issubset({0, 1}), \
        f"y_train contains non-binary labels: {np.unique(y_train_bin)}"

    scaler = StandardScaler()
    Xtr    = scaler.fit_transform(X_train)
    Xte    = scaler.transform(X_test)

    clf    = LogisticRegression(C=C, max_iter=1000, random_state=seed, solver="lbfgs")
    clf.fit(Xtr, y_train_bin)

    # predict_proba returns (N, 2) — column 1 is P(class=1)
    probs  = clf.predict_proba(Xte)[:, 1].ravel()
    preds  = clf.predict(Xte).ravel()
    coef   = clf.coef_.ravel()[:X_train.shape[1]]
    return preds, probs, coef


logger.info("Utility functions ready — meta-learner: LogisticRegression (L2).")

## 3. Load Static Data (Once, Outside the Loop)

> **Design note:** All LLM arrays are Phase B outputs that never change across seeds.
> They are loaded once here to avoid redundant I/O.
> `llm_train_preds.npy` and `llm_rand_preds.npy` are concatenated into a single
> training array that aligns with the combined OOF encoder pool.
> `llm_test_preds.npy` is kept separate and used only for evaluation.

In [ ]:
# ── Inspect columns first to catch naming mismatches early ───────────────────
_test_df_check = pd.read_csv(TEST_LABELS_CSV)
print("datd_test.csv columns :", _test_df_check.columns.tolist())
print("datd_test.csv shape   :", _test_df_check.shape)

# ── UPDATE LABEL_COL HERE if the printed column name differs from 'label' ─────
LABEL_COL = "label"   # ← adjust to match the actual column name printed above

# Ground-truth test labels — identical across all 30 runs
y_test = _test_df_check[LABEL_COL].dropna().values.astype(np.int32)
logger.info("y_test shape : %s  (pos=%.1f%%)", y_test.shape, 100 * y_test.mean())

# Training labels: concatenate datd_train + datd_rand.
# datd_rand.csv has already been cleaned (2 NaN rows removed),
# so the combined pool is exactly 9181 rows — matching the OOF files.
_train_df = pd.read_csv(TRAIN_LABELS_CSV)[[LABEL_COL]].dropna().reset_index(drop=True)
_rand_df  = pd.read_csv(RAND_LABELS_CSV )[[LABEL_COL]].dropna().reset_index(drop=True)
y_train_full = np.concatenate([
    _train_df[LABEL_COL].values.astype(np.int32),
    _rand_df[LABEL_COL].values.astype(np.int32),
])
logger.info("y_train (train) : %d rows", len(_train_df))
logger.info("y_train (rand)  : %d rows", len(_rand_df))
logger.info("y_train combined: %d rows (pos=%.1f%%)",
            len(y_train_full), 100 * y_train_full.mean())

# ── LLM training array: concatenate train + rand ──────────────────────────────
llm_train_part = np.load(LLM_TRAIN_NPY).astype(np.float32)
llm_rand_part  = np.load(LLM_RAND_NPY ).astype(np.float32)
llm_train_full = np.concatenate([llm_train_part, llm_rand_part], axis=0)

# ── LLM test array: unchanged ─────────────────────────────────────────────────
llm_test_flat  = np.load(LLM_TEST_NPY).astype(np.float32)   # (550,)

logger.info("LLM train part  : %s", llm_train_part.shape)
logger.info("LLM rand  part  : %s", llm_rand_part.shape)
logger.info("LLM train+rand  : %s  (combined)", llm_train_full.shape)
logger.info("LLM test        : %s", llm_test_flat.shape)

# All three training arrays must agree on row count before the loop starts
assert llm_train_full.shape[0] == y_train_full.shape[0], \
    f"LLM train rows ({llm_train_full.shape[0]}) != label rows ({y_train_full.shape[0]})"
assert llm_test_flat.shape[0] == y_test.shape[0], \
    f"LLM test rows ({llm_test_flat.shape[0]}) != y_test rows ({y_test.shape[0]})"

# ── Validate that llm_test_flat is prob(class=1), not raw confidence ───────────
# Phase B summary (phase_b_summary.json) reports F1=0.8024, MCC=0.6122.
# If .npy files store raw confidence (not prob_class1), thresholding at 0.5
# produces near-zero MCC because a class-0 prediction with confidence 0.85
# gets misread as 85%% probability of class 1.
# Fix: re-run Phase B cells 6 onwards — saving logic now stores
# np.where(preds==1, conf, 1-conf) instead of raw confidence.
EXPECTED_MCC  = 0.612244   # from phase_b_summary.json
_llm_pred_chk = (llm_test_flat >= 0.5).astype(np.int32)
_llm_mcc_chk  = float(matthews_corrcoef(y_test, _llm_pred_chk))
if abs(_llm_mcc_chk - EXPECTED_MCC) > 0.05:
    raise ValueError(
        f"llm_test_preds.npy validation FAILED (MCC={_llm_mcc_chk:.4f}, expected ~{EXPECTED_MCC}).\n"
        "The .npy files store raw confidence, not prob(class=1).\n"
        "Re-run Phase B cells 6 onwards — the save logic is now corrected."
    )
logger.info("LLM .npy validation PASSED: MCC=%.4f", _llm_mcc_chk)

# ── Pure LLM baseline (static — compute once) ─────────────────────────────────
llm_pred_test = _llm_pred_chk
llm_metrics   = compute_metrics(y_test, llm_pred_test, llm_test_flat)
logger.info("Pure LLM  F1=%.4f  MCC=%.4f  ECE=%.4f",
            llm_metrics["macro_f1"], llm_metrics["mcc"], llm_metrics["ece"])

## 4. The 30-Seed Loop

In [ ]:
# Storage — one dict per seed, keyed by config name
records = []          # list of dicts — one per seed, all configs
coef_store = []       # Ridge coefficients (4,) per seed — for boxplot
asym_conf_store = []  # (550,) conf arrays — one per seed, for mean reliability diagram
enc_conf_store  = []  # (550,) best-encoder conf arrays — one per seed

# For reliability diagrams we only keep the last seed's outputs
last_seed_data = {}

logger.info("Starting 30-seed loop...")

for i_seed, seed in enumerate(SEEDS):
    # ── 1. Load per-seed CSVs ─────────────────────────────────────────────────
    oof_path  = Path(OOF_TRAIN_TEMPLATE.format(seed=seed))
    test_path = Path(PROBS_TEST_TEMPLATE.format(seed=seed))

    oof_df   = pd.read_csv(oof_path)
    test_df  = pd.read_csv(test_path)

    # ── 2. Extract encoder arrays ─────────────────────────────────────────────
    enc_train = oof_df[ENCODER_COLS].values.astype(np.float32)    # (9181, 3)
    enc_test  = test_df[ENCODER_COLS].values.astype(np.float32)   # (550,  3)

    # ── 3. Size assertions — must all agree before any computation ─────────────
    assert enc_train.shape[0] == llm_train_full.shape[0] == y_train_full.shape[0], \
        f"[Seed {seed}] Train size mismatch: enc={enc_train.shape[0]}, " \
        f"llm={llm_train_full.shape[0]}, y={y_train_full.shape[0]}"
    assert enc_test.shape[0] == llm_test_flat.shape[0] == y_test.shape[0], \
        f"[Seed {seed}] Test size mismatch."

    # ── 4. Build feature matrices ─────────────────────────────────────────────
    llm_col_train = llm_train_full.reshape(-1, 1)   # (9181, 1)
    llm_col_test  = llm_test_flat.reshape(-1, 1)    # (550,  1)

    X_train_homo = enc_train                                         # (9181, 3)
    X_train_asym = np.hstack([enc_train, llm_col_train])             # (9181, 4)
    X_test_homo  = enc_test                                          # (550,  3)
    X_test_asym  = np.hstack([enc_test,  llm_col_test])              # (550,  4)

    # ── 5. C tuning (per-seed, inside the loop — correct) ─────────────────────
    best_C_homo = tune_C(X_train_homo, y_train_full, seed=seed)
    best_C_asym = tune_C(X_train_asym, y_train_full, seed=seed)

    # ── 6. Fit & predict — Homogeneous ────────────────────────────────────────
    pred_homo, conf_homo, _ = fit_predict_logistic(
        X_train_homo, y_train_full, X_test_homo, best_C_homo, seed
    )

    # ── 7. Fit & predict — Asymmetric ─────────────────────────────────────────
    pred_asym, conf_asym, coef_asym = fit_predict_logistic(
        X_train_asym, y_train_full, X_test_asym, best_C_asym, seed
    )
    coef_store.append(coef_asym)   # (4,) — for boxplot
    asym_conf_store.append(conf_asym)

    # ── 9. Best Single Encoder (oracle — highest MCC among 3 encoders) ────────
    # ⚠ THESIS NOTE: This picks the best encoder *per seed on the test set*.
    # It is an oracle/upper-bound, NOT a deployable model, because you cannot
    # know which encoder is best without observing test labels.
    best_enc_metrics = None
    best_enc_name    = None
    for enc_col in ENCODER_COLS:
        col_conf = test_df[enc_col].values.astype(np.float32)
        col_pred = (col_conf >= 0.5).astype(np.int32)
        m        = compute_metrics(y_test, col_pred, col_conf)
        if best_enc_metrics is None or m["mcc"] > best_enc_metrics["mcc"]:
            best_enc_metrics = m
            best_enc_name    = enc_col

    # ── 10. Compute metrics once per config, then unpack ──────────────────────
    m_homo = compute_metrics(y_test, pred_homo, conf_homo)
    m_asym = compute_metrics(y_test, pred_asym, conf_asym)

    records.append({
        "seed":          seed,
        "best_enc_name": best_enc_name,
        # Best Single Encoder (oracle)
        "enc_macro_f1":  best_enc_metrics["macro_f1"],
        "enc_mcc":       best_enc_metrics["mcc"],
        "enc_ece":       best_enc_metrics["ece"],
        # Homogeneous Stacking
        "homo_macro_f1": m_homo["macro_f1"],
        "homo_mcc":      m_homo["mcc"],
        "homo_ece":      m_homo["ece"],
        # Asymmetric Stacking
        "asym_macro_f1": m_asym["macro_f1"],
        "asym_mcc":      m_asym["mcc"],
        "asym_ece":      m_asym["ece"],
        "alpha_homo":    best_C_homo,
        "alpha_asym":    best_C_asym,
    })


    enc_conf_store.append(test_df[best_enc_name].values.astype(np.float32))

    # Print per-seed ECE
    print(f"Seed {seed:4d} | ECE: enc={best_enc_metrics['ece']:.4f}  "
          f"llm={llm_metrics['ece']:.4f}  "
          f"homo={m_homo['ece']:.4f}  asym={m_asym['ece']:.4f}")
    # Keep last seed's data for reliability diagrams
    if seed == SEEDS[-1]:
        last_seed_data = {
            "best_enc_conf": test_df[best_enc_name].values.astype(np.float32),
            "best_enc_pred": (test_df[best_enc_name].values >= 0.5).astype(np.int32),
            "best_enc_name": best_enc_name,
            "asym_conf":     conf_asym,
            "asym_pred":     pred_asym,
        }

    completed = i_seed + 1
    if completed % 10 == 0 or completed == N_RUNS:
        logger.info("Completed %d/%d seeds", completed, N_RUNS)

logger.info("30-seed loop complete.")

## 5. Aggregate Results

In [ ]:
df = pd.DataFrame(records)
df.to_csv(OUT_DIR / "phase_c_raw_results.csv", index=False)

def agg(col_prefix):
    """Return mean ± std dict for macro_f1, mcc, ece of a given config prefix."""
    out = {}
    for metric in ["macro_f1", "mcc", "ece"]:
        col = f"{col_prefix}_{metric}"
        out[f"{metric}_mean"] = df[col].mean()
        out[f"{metric}_std"]  = df[col].std(ddof=1)
    return out

# Aggregate per configuration
enc_agg  = agg("enc")
homo_agg = agg("homo")
asym_agg = agg("asym")

# Pure LLM is deterministic — zero std
llm_agg = {
    "macro_f1_mean": llm_metrics["macro_f1"], "macro_f1_std": 0.0,
    "mcc_mean":      llm_metrics["mcc"],      "mcc_std":      0.0,
    "ece_mean":      llm_metrics["ece"],       "ece_std":      0.0,
}

configs = [
    ("Best Single Encoder (Oracle ⚠)",        enc_agg),
    ("Pure Zero-Shot LLM (Gemini 3 Flash)",    llm_agg),
    ("Homogeneous Stacking (3 BERTs)",         homo_agg),
    ("Asymmetric Ridge Stacking (Proposed)",   asym_agg),
]

logger.info("Aggregation complete. Raw results saved.")

## 6. Paired T-Tests: Homogeneous vs. Asymmetric

In [ ]:
# One-tailed paired t-test: H1 = Asymmetric > Homogeneous
# Performed on BOTH macro_f1 and mcc for completeness.

def paired_ttest_one_tailed(a, b, label):
    """
    One-tailed paired t-test: H1 — mean(a) > mean(b).
    Returns a result dict and prints a formatted summary.
    """
    t, p_two = stats.ttest_rel(a, b)
    p_one    = p_two / 2 if t > 0 else 1.0 - p_two / 2
    sig      = p_one < P_ALPHA
    delta    = a.mean() - b.mean()
    print(f"  {label}")
    print(f"    Δ mean     : {delta:+.4f}")
    print(f"    t-statistic: {t:.4f}")
    print(f"    p (1-tail) : {p_one:.6f}")
    print(f"    Significant: {'✅ YES' if sig else '❌ NO'} (α={P_ALPHA})")
    return {"metric": label, "delta": round(delta, 6),
            "t_stat": round(t, 6), "p_one_tailed": round(p_one, 6),
            "significant": bool(sig)}

asym_f1   = df["asym_macro_f1"].values
homo_f1   = df["homo_macro_f1"].values
asym_mcc  = df["asym_mcc"].values
homo_mcc  = df["homo_mcc"].values

divider = "=" * 58
print(divider)
print("  PAIRED T-TEST: Asymmetric vs. Homogeneous Stacking")
print(f"  H1: Asymmetric > Homogeneous  |  N={N_RUNS} seeds")
print(divider)
ttest_f1  = paired_ttest_one_tailed(asym_f1,  homo_f1,  "Macro F1")
print()
ttest_mcc = paired_ttest_one_tailed(asym_mcc, homo_mcc, "MCC")
print(divider)

ttest_results = [ttest_f1, ttest_mcc]

## 7. Summary Table (Thesis-Ready)

In [ ]:
def fmt(mean, std, ndigits=4):
    if std == 0.0:
        return f"{mean:.{ndigits}f}"
    return f"{mean:.{ndigits}f} ± {std:.{ndigits}f}"

table_rows = []
for i, (name, agg) in enumerate(configs, 1):
    table_rows.append([
        i, name,
        fmt(agg["macro_f1_mean"], agg["macro_f1_std"]),
        fmt(agg["mcc_mean"],      agg["mcc_std"]),
        fmt(agg["ece_mean"],      agg["ece_std"]),
    ])

headers = ["#", "Configuration", "Macro F1", "MCC", "ECE (↓)"]
print("\n" + tabulate(table_rows, headers=headers, tablefmt="github"))
print()
print("  ⚠  Row 1 (Best Single Encoder) is an oracle baseline — it selects the")
print("     best encoder per seed using test-set MCC. It is NOT deployable and")
print("     should be labelled as such in the thesis.")

# Save CSV version
pd.DataFrame(table_rows, columns=headers).to_csv(
    OUT_DIR / "phase_c_summary_table.csv", index=False
)
logger.info("Summary table saved.")

## 8. Visualization A — Ridge Coefficient Boxplot (Weight Analysis)

In [ ]:
coef_matrix = np.array(coef_store)   # (30, 4)

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(
    coef_matrix,
    labels=FEATURE_NAMES,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker="o", markersize=4, alpha=0.5),
)

# Color encoders blue, LLM orange
colors = ["steelblue"] * len(ENCODER_COLS) + ["darkorange"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title(
    f"Asymmetric Ridge Meta-Learner — Coefficient Distribution ({N_RUNS} seeds)\n"
    "Blue = BERT encoders  |  Orange = Gemini 3 Flash",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("Ridge Coefficient", fontsize=13)
ax.set_xlabel("Feature", fontsize=13)
ax.grid(axis="y", alpha=0.3)

legend_handles = [
    mpatches.Patch(facecolor="steelblue",  alpha=0.75, label="Fine-tuned Encoder"),
    mpatches.Patch(facecolor="darkorange", alpha=0.75, label="Gemini 3 Flash (LLM)"),
]
ax.legend(handles=legend_handles, fontsize=11)
plt.tight_layout()

boxplot_path = OUT_DIR / "phase_c_coef_boxplot.png"
plt.savefig(boxplot_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info("Saved: %s", boxplot_path)

## 9. Visualization B — Reliability Diagrams (Calibration Map)

In [ ]:
def reliability_diagram(ax, y_true, y_conf, y_pred, title, color, n_bins=ECE_BINS):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []

    for low, high in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_conf >= low) & (y_conf <= high) if high == 1.0 \
               else (y_conf >= low) & (y_conf < high)
        if mask.sum() > 0:
            bin_accs.append((y_pred[mask] == y_true[mask]).mean())
            bin_confs.append(y_conf[mask].mean())
            bin_sizes.append(mask.sum())

    ece_val = expected_calibration_error(y_true, y_conf, y_pred)

    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect")
    if bin_confs:
        sc = ax.scatter(bin_confs, bin_accs, c=bin_sizes,
                        cmap="YlOrRd", s=80, zorder=5, label="Bins")
        ax.plot(bin_confs, bin_accs, "-", color=color, linewidth=1.5)
        plt.colorbar(sc, ax=ax, label="Bin size")

    ax.set_title(f"{title}\nECE = {ece_val:.4f}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Mean Confidence")
    ax.set_ylabel("Fraction Correct")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    f"Reliability Diagrams — Seed {SEEDS[-1]} (last run)",
    fontsize=14, fontweight="bold"
)

reliability_diagram(
    axes[0],
    y_test,
    last_seed_data["best_enc_conf"],
    last_seed_data["best_enc_pred"],
    title=f"Best Single Encoder\n({last_seed_data['best_enc_name']})",
    color="steelblue",
)
reliability_diagram(
    axes[1],
    y_test,
    last_seed_data["asym_conf"],
    last_seed_data["asym_pred"],
    title="Asymmetric Ridge Stacking\n(Proposed)",
    color="forestgreen",
)

plt.tight_layout()
rel_path = OUT_DIR / "phase_c_reliability_diagrams.png"
plt.savefig(rel_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info("Saved: %s", rel_path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualization D — Mean Reliability Diagram (averaged across all 30 seeds)
#
# Strategy: for each bin we pool acc and conf values from every seed,
# then compute the weighted (by total bin samples) mean accuracy and
# mean confidence.  This avoids the artefact of averaging ECE scalars
# and instead gives a calibration picture that reflects the full 30-run
# distribution.
# ─────────────────────────────────────────────────────────────────────────────

def mean_reliability_diagram(ax, y_true, conf_store, n_bins=ECE_BINS,
                              color="steelblue", label="Model"):
    """
    Plot a reliability diagram averaged across all seeds.

    conf_store : list of (N,) float arrays  — one per seed
    y_true     : (N,) int array             — fixed ground truth

    For each bin we accumulate (n_correct, n_total, sum_conf) across
    all seeds, then compute:
        mean_acc  = total_correct / total_samples
        mean_conf = total_conf_sum / total_samples
    The shaded band shows ±1 std of per-seed bin accuracies.
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    n_seeds   = len(conf_store)

    # Per-seed bin accuracies and confidences for the band
    per_seed_acc  = np.full((n_seeds, n_bins), np.nan)
    per_seed_conf = np.full((n_seeds, n_bins), np.nan)

    # Aggregate totals for the pooled curve
    total_correct  = np.zeros(n_bins)
    total_conf_sum = np.zeros(n_bins)
    total_samples  = np.zeros(n_bins)

    for s_idx, y_conf in enumerate(conf_store):
        y_pred = (y_conf >= 0.5).astype(np.int32)
        for b, (low, high) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
            mask = (y_conf >= low) & (y_conf <= high) if high == 1.0 \
                   else (y_conf >= low) & (y_conf < high)
            n = mask.sum()
            if n == 0:
                continue
            per_seed_acc [s_idx, b] = (y_pred[mask] == y_true[mask]).mean()
            per_seed_conf[s_idx, b] = y_conf[mask].mean()
            total_correct [b] += (y_pred[mask] == y_true[mask]).sum()
            total_conf_sum[b] += y_conf[mask].sum()
            total_samples [b] += n

    # Pooled mean curve
    active = total_samples > 0
    mean_acc  = np.where(active, total_correct  / total_samples, np.nan)
    mean_conf = np.where(active, total_conf_sum / total_samples, np.nan)

    # ±1 std band from per-seed values
    std_acc = np.nanstd(per_seed_acc, axis=0, ddof=1)

    # Mean ECE (scalar) — average of per-seed ECEs
    ece_vals = [
        expected_calibration_error(y_true, yc, (yc >= 0.5).astype(np.int32))
        for yc in conf_store
    ]
    mean_ece = np.mean(ece_vals)
    std_ece  = np.std(ece_vals, ddof=1)

    # ── Plot ──────────────────────────────────────────────────────────────────
    x = mean_conf[active]
    y = mean_acc [active]
    e = std_acc  [active]

    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1.2,
            label="Perfect calibration", zorder=1)

    ax.fill_between(x, np.clip(y - e, 0, 1), np.clip(y + e, 0, 1),
                    alpha=0.18, color=color, label="±1 std (across seeds)")

    ax.plot(x, y, "-o", color=color, linewidth=2, markersize=6,
            label=label, zorder=3)

    # Gap fill (miscalibration area)
    ax.fill_between(x, x, y, where=(y < x), alpha=0.12,
                    color="red",  label="Under-confident")
    ax.fill_between(x, x, y, where=(y > x), alpha=0.12,
                    color="blue", label="Over-confident")

    ax.set_title(
        f"{label}\n"
        f"Mean ECE = {mean_ece:.4f} ± {std_ece:.4f}  (N={n_seeds} seeds)",
        fontsize=13, fontweight="bold"
    )
    ax.set_xlabel("Mean Predicted Confidence", fontsize=12)
    ax.set_ylabel("Fraction Correct",          fontsize=12)
    ax.set_xlim(0, 1);  ax.set_ylim(0, 1)
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(alpha=0.25)
    return mean_ece, std_ece


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(
    f"Mean Reliability Diagram — All {N_RUNS} Seeds (2026–2055)",
    fontsize=15, fontweight="bold"
)

mean_reliability_diagram(
    axes[0], y_test, enc_conf_store,
    color="steelblue",
    label="Best Single Encoder (Oracle ⚠)",
)
mean_reliability_diagram(
    axes[1], y_test, asym_conf_store,
    color="forestgreen",
    label="Asymmetric Ridge Stacking (Proposed)",
)

plt.tight_layout()
mean_rel_path = OUT_DIR / "phase_c_mean_reliability_diagrams.png"
plt.savefig(mean_rel_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info("Saved: %s", mean_rel_path)


## 10. Visualization C — Per-Metric Comparison Across Seeds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Phase C — Configuration Comparison (Mean ± Std, 30 Seeds)",
             fontsize=14, fontweight="bold")

cfg_labels = [name.replace(" (", "\n(") for name, _ in configs]
bar_colors = ["#7f8c8d", "#e67e22", "#9b59b6", "#27ae60"]

metric_specs = [
    ("macro_f1", "Macro F1-Score",  True),
    ("mcc",      "MCC",             True),
    ("ece",      "ECE (↓ better)",  False),
]

for ax, (key, ylabel, higher_better) in zip(axes, metric_specs):
    means = [agg[f"{key}_mean"] for _, agg in configs]
    stds  = [agg[f"{key}_std"]  for _, agg in configs]
    x     = np.arange(len(configs))

    bars = ax.bar(x, means, yerr=stds, capsize=4, color=bar_colors,
                  edgecolor="white", width=0.6,
                  error_kw={"linewidth": 1.5, "ecolor": "#333"})

    # Gold border on best (skip oracle for fairness)
    compare_means = [(i, m) for i, m in enumerate(means) if i > 0]
    best_i = max(compare_means, key=lambda t: t[1] if higher_better else -t[1])[0]
    bars[best_i].set_edgecolor("gold")
    bars[best_i].set_linewidth(2.5)

    ax.set_xticks(x)
    ax.set_xticklabels(cfg_labels, fontsize=9, rotation=20, ha="right")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(ylabel, fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
cmp_path = OUT_DIR / "phase_c_comparison_chart.png"
plt.savefig(cmp_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info("Saved: %s", cmp_path)

## 11. Save Full JSON Summary

In [ ]:
coef_matrix_np = np.array(coef_store)   # (30, 4)

summary = {
    "phase":       "C — Asymmetric Ensemble Synthesis (LogisticRegression)",
    "meta_learner": "LogisticRegression (L2, lbfgs)",
    "seeds":       [int(s) for s in SEEDS],
    "n_runs":      N_RUNS,
    "cv_folds":    CV_FOLDS,
    "C_grid":      C_GRID,
    "paired_ttest": ttest_results,
    "configurations": {
        name: {
            k: round(float(v), 6) for k, v in agg.items()
        }
        for name, agg in configs
    },
    "ridge_coef_mean": {
        name: round(float(v), 6)
        for name, v in zip(FEATURE_NAMES, coef_matrix_np.mean(axis=0))
    },
    "ridge_coef_std": {
        name: round(float(v), 6)
        for name, v in zip(FEATURE_NAMES, coef_matrix_np.std(axis=0, ddof=1))
    },
    "output_files": {
        "raw_results":       str(OUT_DIR / "phase_c_raw_results.csv"),
        "summary_table":     str(OUT_DIR / "phase_c_summary_table.csv"),
        "coef_boxplot":      str(OUT_DIR / "phase_c_coef_boxplot.png"),
        "reliability":       str(OUT_DIR / "phase_c_reliability_diagrams.png"),
        "comparison_chart":  str(OUT_DIR / "phase_c_comparison_chart.png"),
    }
}

summary_path = OUT_DIR / "phase_c_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
logger.info("Summary saved: %s", summary_path)

## 12. Output Files Inventory

In [ ]:
output_files = [
    (OUT_DIR / "phase_c_raw_results.csv",         "Per-seed raw metrics for all configs"),
    (OUT_DIR / "phase_c_summary_table.csv",        "Thesis-ready Mean ± Std table"),
    (OUT_DIR / "phase_c_summary.json",             "Full JSON summary + t-test + coef stats"),
    (OUT_DIR / "phase_c_coef_boxplot.png",         "Ridge coefficient boxplot (30 seeds)"),
    (OUT_DIR / "phase_c_reliability_diagrams.png", "Reliability diagrams (best enc vs. proposed)"),
    (OUT_DIR / "phase_c_comparison_chart.png",     "F1 / MCC / ECE bar comparison"),
]

print(f"\n📁 Output Directory: {OUT_DIR}")
print("─" * 72)
for fpath, desc in output_files:
    p      = Path(fpath)
    exists = p.is_file()
    size   = f"{p.stat().st_size / 1024:.1f} KB" if exists else "NOT FOUND"
    status = "✅" if exists else "❌"
    print(f"  {status}  {p.name:<50}  {size:<12}  {desc}")
print("─" * 72)
print("\n✅ Phase C complete.")